<a href="https://colab.research.google.com/github/Nikitoniy/Waka_projects/blob/main/%D0%9F%D0%BE%D0%B8%D1%81%D0%BA_%D0%BC%D0%B5%D1%82%D1%80%D0%BE_%D0%B8_%D1%81%D1%82%D0%BE%D1%80%D0%BE%D0%BD_%D1%81%D0%B2%D0%B5%D1%82%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Географический анализ торговых точек

**Описание проекта**

Проект направлен на обработку адресов торговых точек и определение их географического положения относительно ближайших станций метро и центра Москвы.

Исходные данные содержат адреса торговых точек, которые могут быть записаны в разных форматах и требовать предварительной нормализации. Для дальнейшего географического анализа необходимо привести адреса к единому виду, получить географические координаты и на их основе определить ближайшую станцию метро и направление расположения каждой точки относительно центра города.

Для автоматизации этих этапов были использованы сервисы **DaData** для нормализации адресов и **Яндекс Геокодер** для получения координат.

**Основные этапы обработки:**

1. **Нормализация адресов.** Обработка исходных адресов с использованием сервиса DaData для приведения их к единому формату и повышения согласованности данных.

2. **Геокодирование адресов.** Получение географических координат — широты и долготы — для каждой торговой точки с помощью API Яндекс Геокодера.

3. **Подготовка данных о станциях метро.** Использование координат станций метро для последующего расчёта расстояний от торговых точек до ближайших станций.

4. **Поиск ближайшей станции метро.** Расчёт расстояний между координатами торговых точек и станций метро с использованием формулы гаверсинуса. На основании полученных значений для каждой точки определяется ближайшая станция метро.

5. **Определение стороны света.** Определение направления расположения торговых точек относительно выбранной центральной точки Москвы на основе географических координат.

6. **Формирование итоговой таблицы.** Объединение результатов обработки в единую таблицу с нормализованными адресами, координатами, ближайшими станциями метро и направлением расположения торговых точек.

## Подготовка данных

### Библиотеки

In [22]:
!pip install dadata openpyxl -q

In [23]:
import math
from time import sleep
import numpy as np
import pandas as pd
import requests
from dadata import Dadata
from tqdm.auto import tqdm
from getpass import getpass
from google.colab import files

### Данные для работы

In [24]:
#dadata
token = getpass('Dadata token: ')
secret = getpass('Dadata secret: ')

dadata = Dadata(token, secret)

#geocoder
api_key = getpass('Geocoder api: ')
api_url = 'https://geocode-maps.yandex.ru/v1/'

df = pd.read_excel('shops.xlsx', sheet_name='main')
df.columns = ['name']

sub_df = pd.read_excel('sub.xlsx', sheet_name='msc')

df.info()

Dadata token: ··········
Dadata secret: ··········
Geocoder api: ··········
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    10 non-null     object
dtypes: object(1)
memory usage: 212.0+ bytes


## Нормализация адресов

In [25]:
#убираю дубликаты
unique_addresses = (
                     df['name']
                     .dropna()
                     .astype(str)
                     .str.strip()
                     .str.lower()
                     .unique()
                   )

address_map = {}

#нормализация
for address in tqdm(unique_addresses):
    try:
        result = dadata.clean('address', address)
        address_map[address] = result.get('result', '') if result else ''
        sleep(0.1)

    except:
        None


#мэппинг
df['normalize_address'] = (
                            df['name']
                            .astype(str)
                            .str.strip()
                            .str.lower()
                            .map(address_map)
                          )
df.head(10)

  0%|          | 0/10 [00:00<?, ?it/s]

,name,normalize_address
0,"Москва, Большая Черёмушкинская улица, 1","г Москва, ул Большая Черёмушкинская, д 1"
1,"Москва, Головинское шоссе, 5А","г Москва, ш Головинское, д 5А"
2,"Москва, Рублёвское шоссе, 62","г Москва, ш Рублёвское, д 62"
3,"Москва, Рождественский бульвар, 1","г Москва, б-р Рождественский, д 1"
4,"Московская область, Раменский городской округ,...","Московская обл, г Раменское, пгт Быково, ул Пр..."
5,"Улица Черняховского, 6/40","г Москва, ул Черняховского, д 6/40"
6,"Сходненская улица, 17","г Москва, ул Сходненская, д 17"
7,"ТЦ Ареал, Алтуфьевское шоссе, 86 к1","г Москва, ш Алтуфьевское, д 86 к 1"
8,"Москва, Профсоюзная улица, 152к1","г Москва, ул Профсоюзная, д 152 к 1"
9,"Москва, Профсоюзная улица, 118","г Москва, ул Профсоюзная, д 118"


## Геокодирование

### Функция геокодирования

In [26]:
def geocode(address):
    try:
        #параметры запроса
        params = {
                  'geocode': address,
                  'apikey': api_key,
                  'format': 'json',
                  'results': 1
                 }

        response = requests.get(api_url, params=params, timeout=10)
        response.raise_for_status()

        #результаты
        data = response.json()
        objects = data['response']['GeoObjectCollection']['featureMember']

        #проверка результата
        if not objects:
            return None, None

        #результаты
        coords = objects[0]['GeoObject']['Point']['pos']
        lon, lat = map(float, coords.split())

        return lat, lon

    except:
        return None, None

### Получение координат

In [27]:
#убираю дубликаты
unique_addresses = (
                     df['normalize_address']
                     .dropna()
                     .unique()
                   )

address_map = {}

#геокодирование
for address in tqdm(unique_addresses):
    try:
        address_map[address] = geocode(address)
        sleep(0.1)

    except:
        None


#мэппинг широты
df['lat'] = (
             df['normalize_address']
             .map(address_map)
             .str[0]
            )

#мэппинг долготы
df['lon'] = (
             df['normalize_address']
             .map(address_map)
             .str[1]
            )

df.head(10)

  0%|          | 0/10 [00:00<?, ?it/s]

,name,normalize_address,lat,lon
0,"Москва, Большая Черёмушкинская улица, 1","г Москва, ул Большая Черёмушкинская, д 1",55.690283,37.601879
1,"Москва, Головинское шоссе, 5А","г Москва, ш Головинское, д 5А",55.839921,37.490209
2,"Москва, Рублёвское шоссе, 62","г Москва, ш Рублёвское, д 62",55.766218,37.380875
3,"Москва, Рождественский бульвар, 1","г Москва, б-р Рождественский, д 1",55.766749,37.624211
4,"Московская область, Раменский городской округ,...","Московская обл, г Раменское, пгт Быково, ул Пр...",55.624993,38.073225
5,"Улица Черняховского, 6/40","г Москва, ул Черняховского, д 6/40",55.803929,37.535152
6,"Сходненская улица, 17","г Москва, ул Сходненская, д 17",55.842044,37.438269
7,"ТЦ Ареал, Алтуфьевское шоссе, 86 к1","г Москва, ш Алтуфьевское, д 86 к 1",55.897791,37.588233
8,"Москва, Профсоюзная улица, 152к1","г Москва, ул Профсоюзная, д 152 к 1",55.622314,37.505894
9,"Москва, Профсоюзная улица, 118","г Москва, ул Профсоюзная, д 118",55.636255,37.521129


## Метро

In [28]:
#расчет расстояния между точками и поиск метро
def haversine(lat1, lon1, lat2, lon2):

    #радиус
    R = 6371000

    #перевожу из градусов в радианы
    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    #расчет для формулы гаверсинусов
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )

    #расчет итоги
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

df['sub'] = None
df['distance_m'] = None

#ищу метро
for idx, row in df.iterrows():

    if pd.isna(row['lat']) or pd.isna(row['lon']):
        continue

    min_dist = float('inf')

    #перебираю все станции метро
    for _, metro in sub_df.iterrows():

        dist = haversine(
                         float(row['lat']),
                         float(row['lon']),
                         float(metro['lat']),
                         float(metro['lon'])
                        )

        if dist < min_dist:
            min_dist = dist
            nearest_metro = metro['name']

    #результаты
    if min_dist != float('inf'):
        df.at[idx, 'sub'] = nearest_metro
        df.at[idx, 'distance_m'] = round(min_dist)

df.head(10)

,name,normalize_address,lat,lon,sub,distance_m
0,"Москва, Большая Черёмушкинская улица, 1","г Москва, ул Большая Черёмушкинская, д 1",55.690283,37.601879,Крымская,444
1,"Москва, Головинское шоссе, 5А","г Москва, ш Головинское, д 5А",55.839921,37.490209,Водный стадион,222
2,"Москва, Рублёвское шоссе, 62","г Москва, ш Рублёвское, д 62",55.766218,37.380875,Крылатское,2005
3,"Москва, Рождественский бульвар, 1","г Москва, б-р Рождественский, д 1",55.766749,37.624211,Трубная,355
4,"Московская область, Раменский городской округ,...","Московская обл, г Раменское, пгт Быково, ул Пр...",55.624993,38.073225,МЦД Быково,999
5,"Улица Черняховского, 6/40","г Москва, ул Черняховского, д 6/40",55.803929,37.535152,Аэропорт,432
6,"Сходненская улица, 17","г Москва, ул Сходненская, д 17",55.842044,37.438269,Сходненская,951
7,"ТЦ Ареал, Алтуфьевское шоссе, 86 к1","г Москва, ш Алтуфьевское, д 86 к 1",55.897791,37.588233,Алтуфьево,86
8,"Москва, Профсоюзная улица, 152к1","г Москва, ул Профсоюзная, д 152 к 1",55.622314,37.505894,Тёплый Стан,382
9,"Москва, Профсоюзная улица, 118","г Москва, ул Профсоюзная, д 118",55.636255,37.521129,Коньково,354


## Сторона света

### Функция для центра

In [29]:
#центр
center_lat = 55.7520
center_lon = 37.6175

#расчет расстояния от точки до центра москвы
def distance_from_center(lat, lon):

    #радиус
    R = 6371000

    #перевожу из градусов в радианы
    lat1 = math.radians(center_lat)
    lon1 = math.radians(center_lon)
    lat2 = math.radians(lat)
    lon2 = math.radians(lon)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    #расчет для формулы гаверсинусов
    a = (
         math.sin(dlat / 2) ** 2
         + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
        )

    #расчет итоги
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

### Определение стороны света и центра

In [30]:
#определение стороны света
def direction(lat, lon):

    #пропускаю пустые строки
    if pd.isna(lat) or pd.isna(lon):
        return None

    #рассчитываю расстояние от адреса до центра
    distance = distance_from_center(lat, lon)

    if distance <= 5000:
        return 'Центр'

    dlat = lat - center_lat
    dlon = (lon - center_lon) * math.cos(math.radians(center_lat))

    #угол направления
    angle = math.degrees(math.atan2(dlon, dlat)) % 360

    #все направления
    directions = [
                  'Север', 'Северо-Восток', 'Восток', 'Юго-Восток',
                  'Юг', 'Юго-Запад', 'Запад', 'Северо-Запад'
                 ]

    #возвращаю сторону света по углу
    return directions[round(angle / 45) % 8]

#результаты
df['side'] = df.apply(
                                lambda row: direction(row['lat'], row['lon']),
                                axis=1
                               )

df.head(10)

,name,normalize_address,lat,lon,sub,distance_m,side
0,"Москва, Большая Черёмушкинская улица, 1","г Москва, ул Большая Черёмушкинская, д 1",55.690283,37.601879,Крымская,444,Юг
1,"Москва, Головинское шоссе, 5А","г Москва, ш Головинское, д 5А",55.839921,37.490209,Водный стадион,222,Северо-Запад
2,"Москва, Рублёвское шоссе, 62","г Москва, ш Рублёвское, д 62",55.766218,37.380875,Крылатское,2005,Запад
3,"Москва, Рождественский бульвар, 1","г Москва, б-р Рождественский, д 1",55.766749,37.624211,Трубная,355,Центр
4,"Московская область, Раменский городской округ,...","Московская обл, г Раменское, пгт Быково, ул Пр...",55.624993,38.073225,МЦД Быково,999,Юго-Восток
5,"Улица Черняховского, 6/40","г Москва, ул Черняховского, д 6/40",55.803929,37.535152,Аэропорт,432,Северо-Запад
6,"Сходненская улица, 17","г Москва, ул Сходненская, д 17",55.842044,37.438269,Сходненская,951,Северо-Запад
7,"ТЦ Ареал, Алтуфьевское шоссе, 86 к1","г Москва, ш Алтуфьевское, д 86 к 1",55.897791,37.588233,Алтуфьево,86,Север
8,"Москва, Профсоюзная улица, 152к1","г Москва, ул Профсоюзная, д 152 к 1",55.622314,37.505894,Тёплый Стан,382,Юго-Запад
9,"Москва, Профсоюзная улица, 118","г Москва, ул Профсоюзная, д 118",55.636255,37.521129,Коньково,354,Юго-Запад


## Сохранение

In [31]:
#df.to_csv('subs_and_sides.csv', index=False, encoding='utf-8-sig')
#files.download('subs_and_sides.csv')